In [ ]:
# ============================================================
# OUTER FROG-LOGO PHASE 2 SUPERVISED MEAN POOLING
# MULTI-PHASE1-CKPT × SINGLE TRAIN TO EPOCH 70 × MULTI-EPOCH EVAL
# ============================================================

import os
import re
import math
import copy
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning as L

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score

from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger


# ============================================================
# 0. seed
# ============================================================
def seed_everything_all(seed: int = 0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    L.seed_everything(seed, workers=True)

    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


# ============================================================
# 1. metadata helpers
# ============================================================
def normalize_uhm_from_dataset_filename(fn: str) -> str:
    """
    UHM18-10842_MS1.npz -> UHM18-10842
    """
    s = os.path.basename(str(fn))
    s = re.sub(r"\.(npz|mzML)$", "", s)
    s = re.sub(r"_MS1$", "", s)
    return s


def frog_from_uhm_sample(uhm_sample: str) -> str:
    """
    UHM18-10842 -> UHM18
    """
    return str(uhm_sample).split("-")[0]


def build_binary_uhm_maps(meta_csv, pos_label="STP1710.7", neg_label="control"):
    df = pd.read_csv(meta_csv, dtype=str)
    df["UHM_sample"] = df["UHM_sample"].astype(str).str.strip()
    df["treatment"] = df["treatment"].astype(str).str.strip()

    df = df[df["treatment"].isin([pos_label, neg_label])].copy()
    df["y"] = (df["treatment"] == pos_label).astype(int)

    uhm2label = dict(zip(df["UHM_sample"], df["y"]))
    uhm2treat = dict(zip(df["UHM_sample"], df["treatment"]))
    uhm2frog  = {u: frog_from_uhm_sample(u) for u in df["UHM_sample"]}

    return df, uhm2label, uhm2treat, uhm2frog


# ============================================================
# 2. window slicer
# ============================================================
class WindowSlicer:
    def __init__(self, window_size=500, stride=250, jitter_max=0, seed=0):
        self.window_size = int(window_size)
        self.stride = int(stride)
        self.jitter_max = int(jitter_max)
        self.rng = np.random.default_rng(seed)

    def __call__(self, chrom):
        signal = np.asarray(chrom["signal"], dtype=np.float32)
        rt = np.asarray(chrom["rt"], dtype=np.float32)

        Lsig = len(signal)
        ws = self.window_size
        st = self.stride

        if Lsig < ws:
            return []

        starts = list(range(0, Lsig - ws + 1, st))
        out = []

        for s in starts:
            if self.jitter_max > 0:
                j = int(self.rng.integers(-self.jitter_max, self.jitter_max + 1))
                s2 = max(0, min(Lsig - ws, s + j))
            else:
                s2 = s

            e2 = s2 + ws
            out.append({
                "signal": signal[s2:e2].astype(np.float32),
                "rt": rt[s2:e2].astype(np.float32),
            })

        return out


# ============================================================
# 3. phase1 encoder
# ============================================================
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )
        self.register_buffer("div_term", div_term)

    def forward(self, rt):
        """
        rt: (B, T)
        """
        rt = rt.unsqueeze(-1)  # (B, T, 1)
        pe = torch.zeros(rt.size(0), rt.size(1), self.d_model, device=rt.device)
        pe[:, :, 0::2] = torch.sin(rt * self.div_term)
        pe[:, :, 1::2] = torch.cos(rt * self.div_term)
        return pe


class MassSpecWindowContrastEncoder(nn.Module):
    """
    IMPORTANT:
      - signal / 1e6
      - rt_patch / 1800.0
      - CLS token
      - transformer num_layers=1
      - final F.layer_norm on h
    """
    def __init__(self, patch_size=50, stride=50, embed_dim=64, num_heads=4):
        super().__init__()
        self.patch_size = patch_size
        self.stride = stride
        self.embed_dim = embed_dim

        self.conv = nn.Conv1d(
            in_channels=1,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=stride,
            padding=0,
        )

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.positional_encoding = SinusoidalPositionalEncoding(embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=4 * embed_dim,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)

    def forward(self, signal, rt):
        """
        signal, rt: (B, L)
        """
        B = signal.size(0)

        signal = signal / 1e6

        x = self.conv(signal.unsqueeze(1))   # (B, D, T)
        x = x.permute(0, 2, 1)               # (B, T, D)

        rt_patch = rt[:, self.patch_size - 1::self.stride]
        assert x.size(1) == rt_patch.size(1), (
            f"Patch/RT mismatch: conv patches={x.size(1)} vs rt patches={rt_patch.size(1)}"
        )

        rt_patch = rt_patch / 1800.0
        pe = self.positional_encoding(rt_patch)

        x = x + pe
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)

        x = self.transformer(x)
        h = x[:, 0, :]
        h = F.layer_norm(h, h.shape[-1:])
        return h


def strip_prefix_any(sd, prefixes):
    for p in prefixes:
        if p == "":
            return sd, p
        if any(k.startswith(p) for k in sd.keys()):
            out = {k[len(p):]: v for k, v in sd.items() if k.startswith(p)}
            return out, p
    return None, None


def load_phase1_encoder_from_ckpt(
    ckpt_path,
    patch_size=50,
    stride=50,
    embed_dim=64,
    num_heads=4,
):
    model = MassSpecWindowContrastEncoder(
        patch_size=patch_size,
        stride=stride,
        embed_dim=embed_dim,
        num_heads=num_heads,
    )

    ckpt = torch.load(ckpt_path, map_location="cpu")
    sd = ckpt.get("state_dict", ckpt)

    prefixes = ["encoder.", "enc_with_head.encoder.", ""]
    sub, used = strip_prefix_any(sd, prefixes)
    if sub is None:
        raise RuntimeError(f"Could not find encoder prefix in checkpoint: {ckpt_path}")

    missing, unexpected = model.load_state_dict(sub, strict=False)
    print(f"[load ok] ckpt={ckpt_path}")
    print(f"[load ok] prefix={used!r} | missing={len(missing)} | unexpected={len(unexpected)}")
    if len(missing) > 0:
        print("missing example:", missing[:10])
    if len(unexpected) > 0:
        print("unexpected example:", unexpected[:10])

    return model


# ============================================================
# 4. phase2 dataset
# ============================================================
class Phase2SetDataset(Dataset):
    """
    one sample = one file
    read full chromatogram -> slice windows on the fly
    """
    def __init__(
        self,
        npz_files,
        uhm2label,
        uhm2frog,
        window_slicer,
        n_windows_target=32,
        seed=0,
        signal_key="signal_grid",
        rt_key="rt_grid",
    ):
        self.npz_files = [Path(p) for p in npz_files]
        self.uhm2label = uhm2label
        self.uhm2frog = uhm2frog
        self.window_slicer = window_slicer
        self.n_windows_target = n_windows_target
        self.signal_key = signal_key
        self.rt_key = rt_key
        self.rng = np.random.default_rng(seed)

        self.samples = []
        for p in self.npz_files:
            uhm = normalize_uhm_from_dataset_filename(p.name)
            if uhm not in self.uhm2label:
                continue
            self.samples.append({
                "path": p,
                "file_name": p.name,
                "uhm_sample": uhm,
                "frog_id": self.uhm2frog[uhm],
                "y": int(self.uhm2label[uhm]),
            })

    def __len__(self):
        return len(self.samples)

    def _load_chrom(self, npz_path: Path):
        d = np.load(npz_path, allow_pickle=True)
        signal = np.asarray(d[self.signal_key], dtype=np.float32)
        rt = np.asarray(d[self.rt_key], dtype=np.float32)

        if signal.ndim != 1 or rt.ndim != 1:
            raise ValueError(
                f"{npz_path.name}: expected 1D arrays, got signal.shape={signal.shape}, rt.shape={rt.shape}"
            )
        if len(signal) != len(rt):
            raise ValueError(f"{npz_path.name}: len(signal) != len(rt)")

        return {
            "signal": signal,
            "rt": rt,
            "chrom_name": npz_path.name,
        }

    def _sample_windows(self, windows):
        n = len(windows)
        k = self.n_windows_target
        if k is None:
            return windows
        if n == 0:
            return []

        if n >= k:
            idx = self.rng.choice(n, size=k, replace=False)
        else:
            idx = self.rng.choice(n, size=k, replace=True)

        return [windows[i] for i in idx]

    def __getitem__(self, idx):
        item = self.samples[idx]
        chrom = self._load_chrom(item["path"])

        windows = self.window_slicer(chrom)
        windows = self._sample_windows(windows)

        if len(windows) == 0:
            raise RuntimeError(f"No windows generated for {item['file_name']}")

        signal = np.stack([w["signal"] for w in windows], axis=0).astype(np.float32)
        rt     = np.stack([w["rt"]     for w in windows], axis=0).astype(np.float32)
        rt_center = rt.mean(axis=1).astype(np.float32)

        order = np.argsort(rt_center)
        signal = signal[order]
        rt = rt[order]
        rt_center = rt_center[order]

        return {
            "signal": torch.from_numpy(signal).float(),
            "rt": torch.from_numpy(rt).float(),
            "rt_center": torch.from_numpy(rt_center).float(),
            "y": torch.tensor(item["y"], dtype=torch.long),
            "file_name": item["file_name"],
            "uhm_sample": item["uhm_sample"],
            "frog_id": item["frog_id"],
        }


def collate_phase2_set(batch):
    B = len(batch)
    lengths = [b["signal"].shape[0] for b in batch]
    maxN = max(lengths)
    Lsig = batch[0]["signal"].shape[1]

    signal = torch.zeros(B, maxN, Lsig, dtype=torch.float32)
    rt = torch.zeros(B, maxN, Lsig, dtype=torch.float32)
    rt_center = torch.zeros(B, maxN, dtype=torch.float32)
    pad_mask = torch.ones(B, maxN, dtype=torch.bool)   # True = pad
    y = torch.zeros(B, dtype=torch.long)

    file_names, uhm_samples, frog_ids = [], [], []

    for i, b in enumerate(batch):
        n = b["signal"].shape[0]
        signal[i, :n] = b["signal"]
        rt[i, :n] = b["rt"]
        rt_center[i, :n] = b["rt_center"]
        pad_mask[i, :n] = False
        y[i] = b["y"]

        file_names.append(b["file_name"])
        uhm_samples.append(b["uhm_sample"])
        frog_ids.append(b["frog_id"])

    return {
        "signal": signal,
        "rt": rt,
        "rt_center": rt_center,
        "pad_mask": pad_mask,
        "y": y,
        "file_name": file_names,
        "uhm_sample": uhm_samples,
        "frog_id": frog_ids,
    }


# ============================================================
# 5. mean pooling classifier
# ============================================================
class MeanPoolingClassifier(nn.Module):
    """
    Mean pooling baseline:
    window embeddings -> masked mean -> LayerNorm -> linear head
    """
    def __init__(
        self,
        embed_dim=64,
        num_classes=2,
        use_rt_pos=False,
    ):
        super().__init__()
        self.use_rt_pos = use_rt_pos
        self.set_pos = SinusoidalPositionalEncoding(embed_dim)
        self.out_norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, z, rt_center=None, key_padding_mask=None, return_embedding=False):
        x = z

        if self.use_rt_pos and rt_center is not None:
            x = x + self.set_pos(rt_center / 1800.0)

        if key_padding_mask is None:
            h = x.mean(dim=1)
        else:
            valid = (~key_padding_mask).unsqueeze(-1).float()
            denom = valid.sum(dim=1).clamp_min(1.0)
            h = (x * valid).sum(dim=1) / denom

        h = self.out_norm(h)
        logits = self.head(h)

        if return_embedding:
            return h, logits
        return logits


# ============================================================
# 6. lightning module
# ============================================================
class Phase2SetClassifier(L.LightningModule):
    def __init__(
        self,
        window_encoder,
        set_encoder,
        lr=1e-4,
        weight_decay=1e-4,
        freeze_window_encoder=True,
    ):
        super().__init__()
        self.window_encoder = window_encoder
        self.set_encoder = set_encoder
        self.lr = lr
        self.weight_decay = weight_decay
        self.freeze_window_encoder = freeze_window_encoder

        if self.freeze_window_encoder:
            for p in self.window_encoder.parameters():
                p.requires_grad = False
            self.window_encoder.eval()

    def encode_windows(self, signal, rt):
        B, N, Lsig = signal.shape
        signal2 = signal.reshape(B * N, Lsig)
        rt2 = rt.reshape(B * N, Lsig)

        if self.freeze_window_encoder:
            with torch.no_grad():
                z = self.window_encoder(signal2, rt2)
        else:
            z = self.window_encoder(signal2, rt2)

        D = z.shape[-1]
        z = z.reshape(B, N, D)
        return z

    def forward(self, signal, rt, rt_center, pad_mask=None, return_embedding=False):
        z = self.encode_windows(signal, rt)

        if return_embedding:
            h, logits = self.set_encoder(
                z,
                rt_center=rt_center,
                key_padding_mask=pad_mask,
                return_embedding=True,
            )
            return h, logits

        logits = self.set_encoder(
            z,
            rt_center=rt_center,
            key_padding_mask=pad_mask,
            return_embedding=False,
        )
        return logits

    def training_step(self, batch, batch_idx):
        signal = batch["signal"]
        rt = batch["rt"]
        rt_center = batch["rt_center"]
        pad_mask = batch["pad_mask"]
        y = batch["y"]

        logits = self(signal, rt, rt_center, pad_mask=pad_mask)
        loss = F.cross_entropy(logits, y)

        pred = logits.argmax(dim=1)
        acc = (pred == y).float().mean()
        bs = y.shape[0]

        self.log("train/loss", loss, prog_bar=True, batch_size=bs)
        self.log("train/acc", acc, prog_bar=True, batch_size=bs)
        return loss

    def configure_optimizers(self):
        params = [p for p in self.parameters() if p.requires_grad]
        return torch.optim.AdamW(params, lr=self.lr, weight_decay=self.weight_decay)


# ============================================================
# 7. helpers
# ============================================================
def group_files_by_frog(npz_files):
    frog2files = {}
    for p in npz_files:
        uhm = normalize_uhm_from_dataset_filename(p.name)
        frog = frog_from_uhm_sample(uhm)
        frog2files.setdefault(frog, []).append(p)
    return frog2files


def aggregate_file_rows_to_frog(rows):
    frog_rows = []
    df = pd.DataFrame([{
        "file_name": r["file_name"],
        "uhm_sample": r["uhm_sample"],
        "frog_id": r["frog_id"],
        "y": r["y"],
        "prob": r["prob"],
    } for r in rows])

    emb_map = {}
    for r in rows:
        emb_map.setdefault(r["frog_id"], []).append(r["emb"])

    for frog in sorted(df["frog_id"].unique()):
        sub = df[df["frog_id"] == frog]
        prob_mean = float(sub["prob"].mean())
        y = int(round(sub["y"].mean()))
        pred = int(prob_mean >= 0.5)
        emb_mean = np.mean(np.stack(emb_map[frog], axis=0), axis=0)

        frog_rows.append({
            "frog_id": frog,
            "y": y,
            "prob": prob_mean,
            "pred": pred,
            "n_files": len(sub),
            "emb": emb_mean,
        })

    return frog_rows


def compute_binary_metrics_from_frog_rows(frog_rows):
    y_true = np.array([r["y"] for r in frog_rows], dtype=int)
    y_prob = np.array([r["prob"] for r in frog_rows], dtype=float)
    y_pred = np.array([r["pred"] for r in frog_rows], dtype=int)

    return {
        "ACC": accuracy_score(y_true, y_pred),
        "AUROC": roc_auc_score(y_true, y_prob),
        "AUPRC": average_precision_score(y_true, y_prob),
        "n_frogs": len(y_true),
    }


def summarize_mean_std_ci95(df, group_col="epoch", metric_cols=("ACC", "AUROC", "AUPRC")):
    rows = []
    grouped = df.groupby(group_col, dropna=False)

    for key, sub in grouped:
        row = {group_col: key, "n_phase1_ckpts": len(sub)}
        for m in metric_cols:
            vals = sub[m].astype(float).values
            n = len(vals)
            mean = float(np.mean(vals))
            std = float(np.std(vals, ddof=1)) if n > 1 else 0.0
            se = std / np.sqrt(n) if n > 1 else 0.0
            ci_low = mean - 1.96 * se
            ci_high = mean + 1.96 * se

            row[f"{m}_mean"] = mean
            row[f"{m}_std"] = std
            row[f"{m}_se"] = se
            row[f"{m}_ci_low"] = ci_low
            row[f"{m}_ci_high"] = ci_high
        rows.append(row)

    return pd.DataFrame(rows).sort_values(group_col).reset_index(drop=True)


def make_phase1_ckpt_id(ckpt_path):
    p = Path(ckpt_path)
    stem = p.stem
    parent = p.parent.parent.parent.name if len(p.parents) >= 3 else "unknown_exp"
    return f"{parent}__{stem}"


def find_saved_epoch_ckpts(ckpt_dir, eval_epochs):
    """
    Your current Lightning filename pattern is:
        {epoch}-{step}.ckpt
    so examples are:
        9-70.ckpt, 19-140.ckpt, ..., 69-490.ckpt

    Note:
        Lightning epoch is 0-based internally.
        So:
            target epoch 10 -> file starts with "9-"
            target epoch 20 -> file starts with "19-"
            ...
    """
    ckpt_dir = Path(ckpt_dir)
    found = {}

    for ep in eval_epochs:
        zero_based = ep - 1
        matches = sorted(ckpt_dir.glob(f"{zero_based}-*.ckpt"))
        if len(matches) == 0:
            raise FileNotFoundError(
                f"Could not find checkpoint for target epoch={ep} "
                f"(expected pattern {zero_based}-*.ckpt) under {ckpt_dir}"
            )
        found[ep] = str(matches[-1])

    return found


def extract_logged_train_loss_by_epoch(log_dir, eval_epochs):
    """
    Read Lightning CSVLogger metrics.csv and return mean training loss for each target epoch.

    In this notebook, EVAL_EPOCHS are 1-based training epochs (10, 20, ...),
    while Lightning's `epoch` column is 0-based. Therefore epoch 10 maps to
    metrics rows with epoch == 9.
    """
    metrics_path = Path(log_dir) / "metrics.csv"
    loss_by_epoch = {int(ep): np.nan for ep in eval_epochs}

    if not metrics_path.exists():
        print(f"WARNING: metrics.csv not found: {metrics_path}")
        return loss_by_epoch

    dfm = pd.read_csv(metrics_path)
    if "epoch" not in dfm.columns:
        print(f"WARNING: no epoch column in {metrics_path}")
        return loss_by_epoch

    # Lightning CSVLogger column names vary slightly by version/settings.
    candidate_cols = [
        "train/loss_epoch",
        "train/loss",
        "train/loss_step",
    ]
    loss_cols = [c for c in candidate_cols if c in dfm.columns]

    if not loss_cols:
        print(f"WARNING: no train loss column found in {metrics_path}; columns={list(dfm.columns)}")
        return loss_by_epoch

    for ep in eval_epochs:
        zero_based = int(ep) - 1
        g = dfm[dfm["epoch"] == zero_based]

        vals = []
        for c in loss_cols:
            vals.extend(pd.to_numeric(g[c], errors="coerce").dropna().tolist())

        if len(vals) > 0:
            loss_by_epoch[int(ep)] = float(np.mean(vals))

    return loss_by_epoch


# ============================================================
# 8. prediction
# ============================================================
@torch.no_grad()
def predict_on_dataset(model, dataset, pos_class_index=1, device="cuda"):
    loader = DataLoader(
        dataset,
        batch_size=4,
        shuffle=False,
        num_workers=0,
        collate_fn=collate_phase2_set,
    )

    device = torch.device(device if torch.cuda.is_available() else "cpu")
    model = model.to(device).eval()

    rows = []
    for batch in loader:
        signal = batch["signal"].to(device)
        rt = batch["rt"].to(device)
        rt_center = batch["rt_center"].to(device)
        pad_mask = batch["pad_mask"].to(device)

        h, logits = model(signal, rt, rt_center, pad_mask=pad_mask, return_embedding=True)

        prob_all = torch.softmax(logits, dim=1).cpu().numpy()
        prob = prob_all[:, pos_class_index]
        pred = (prob >= 0.5).astype(int)
        emb = h.cpu().numpy()

        y = batch["y"].numpy()
        for i in range(len(y)):
            rows.append({
                "file_name": batch["file_name"][i],
                "uhm_sample": batch["uhm_sample"][i],
                "frog_id": batch["frog_id"][i],
                "y": int(y[i]),
                "prob": float(prob[i]),
                "pred": int(pred[i]),
                "emb": emb[i],
            })
    return rows


# ============================================================
# 9. one outer fold: train once, eval many epochs
# ============================================================
def run_one_outer_fold_multi_epoch(
    outer_test_frog,
    use_files,
    uhm2label,
    uhm2treat,
    uhm2frog,
    phase1_ckpt,
    log_root,
    eval_epochs,
    seed=0,
    window_size=500,
    window_stride=250,
    n_windows_target=32,
    phase1_patch_size=50,
    phase1_patch_stride=50,
    embed_dim=64,
    num_heads=4,
    batch_size=4,
    lr=1e-4,
    weight_decay=1e-4,
    max_epochs=70,
    freeze_window_encoder=True,
    pos_class_index=1,
):
    frog2files = group_files_by_frog(use_files)
    all_frogs = sorted(frog2files.keys())

    assert outer_test_frog in all_frogs, f"{outer_test_frog} not in frogs"

    train_frogs = [g for g in all_frogs if g != outer_test_frog]

    train_files = []
    for g in train_frogs:
        train_files.extend(frog2files[g])

    test_files = []
    for g in [outer_test_frog]:
        test_files.extend(frog2files[g])

    print("=" * 100)
    print(f"[OUTER TEST FROG] {outer_test_frog}")
    print(f"train frogs: {train_frogs}")
    print(f"test frog: {outer_test_frog}")
    print(f"n train files={len(train_files)} | n test files={len(test_files)}")
    print(f"phase1_ckpt: {phase1_ckpt}")

    train_slicer = WindowSlicer(window_size=window_size, stride=window_stride, jitter_max=0, seed=seed)
    test_slicer  = WindowSlicer(window_size=window_size, stride=window_stride, jitter_max=0, seed=seed)

    train_ds = Phase2SetDataset(
        train_files,
        uhm2label=uhm2label,
        uhm2frog=uhm2frog,
        window_slicer=train_slicer,
        n_windows_target=n_windows_target,
        seed=seed,
    )
    test_ds = Phase2SetDataset(
        test_files,
        uhm2label=uhm2label,
        uhm2frog=uhm2frog,
        window_slicer=test_slicer,
        n_windows_target=n_windows_target,
        seed=seed + 1,
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        collate_fn=collate_phase2_set,
    )

    # ---- build model ONCE for training
    window_encoder_train = load_phase1_encoder_from_ckpt(
        phase1_ckpt,
        patch_size=phase1_patch_size,
        stride=phase1_patch_stride,
        embed_dim=embed_dim,
        num_heads=num_heads,
    )

    set_encoder_train = MeanPoolingClassifier(
        embed_dim=embed_dim,
        num_classes=2,
        use_rt_pos=False,
    )

    model = Phase2SetClassifier(
        window_encoder=window_encoder_train,
        set_encoder=set_encoder_train,
        lr=lr,
        weight_decay=weight_decay,
        freeze_window_encoder=freeze_window_encoder,
    )

    freeze_tag = "frozen" if freeze_window_encoder else "unfrozen"
    phase1_id = make_phase1_ckpt_id(phase1_ckpt)
    fold_name = f"phase2_meanpool_logo_{freeze_tag}__phase1-{phase1_id}__test-{outer_test_frog}__seed{seed}"
    logger = CSVLogger(save_dir=log_root, name=fold_name)

    ckpt_dir = Path(logger.save_dir) / logger.name / "version_0" / "checkpoints"

    # save target epochs
    periodic_ckpt_cb = ModelCheckpoint(
        dirpath=ckpt_dir,
        filename="{epoch}-{step}",
        save_top_k=-1,
        every_n_epochs=10,
        save_on_train_epoch_end=True,
        auto_insert_metric_name=False,
    )

    # also save last
    last_ckpt_cb = ModelCheckpoint(
        dirpath=ckpt_dir,
        filename="last",
        save_last=True,
        save_top_k=0,
        every_n_epochs=1,
        auto_insert_metric_name=False,
    )

    trainer = L.Trainer(
        max_epochs=max_epochs,
        accelerator="auto",
        devices=1,
        logger=logger,
        callbacks=[periodic_ckpt_cb, last_ckpt_cb],
        log_every_n_steps=1,
        enable_progress_bar=True,
        deterministic=True,
    )

    trainer.fit(model, train_loader)

    # Mean training loss per evaluation epoch, read from Lightning CSVLogger.
    # This is added to the output CSVs to match the random-embedding baseline.
    train_loss_by_epoch = extract_logged_train_loss_by_epoch(logger.log_dir, eval_epochs)

    print("ckpt_dir:", ckpt_dir)
    saved_ckpts = find_saved_epoch_ckpts(ckpt_dir, eval_epochs)
    print("[saved epoch ckpts]")
    for ep in eval_epochs:
        print(f"  epoch={ep}: {saved_ckpts[ep]}")

    # ---- evaluate each epoch ckpt separately
    epoch_outputs = {}

    for ep in eval_epochs:
        selected_ckpt = saved_ckpts[ep]
        print("\n" + "-" * 100)
        print(f"[EVAL] outer_test_frog={outer_test_frog} | epoch={ep}")
        print(f"selected_ckpt={selected_ckpt}")

        # IMPORTANT: new fresh model objects for each checkpoint load
        window_encoder_eval = load_phase1_encoder_from_ckpt(
            phase1_ckpt,
            patch_size=phase1_patch_size,
            stride=phase1_patch_stride,
            embed_dim=embed_dim,
            num_heads=num_heads,
        )

        set_encoder_eval = MeanPoolingClassifier(
            embed_dim=embed_dim,
            num_classes=2,
            use_rt_pos=False,
        )

        best_model = Phase2SetClassifier.load_from_checkpoint(
            selected_ckpt,
            window_encoder=window_encoder_eval,
            set_encoder=set_encoder_eval,
            lr=lr,
            weight_decay=weight_decay,
            freeze_window_encoder=freeze_window_encoder,
        )

        test_rows = predict_on_dataset(
            best_model,
            test_ds,
            pos_class_index=pos_class_index,
            device="cuda",
        )
        frog_rows = aggregate_file_rows_to_frog(test_rows)
        fold_metrics = compute_binary_metrics_from_frog_rows(frog_rows)

        print("[test frog rows]")
        for r in frog_rows:
            print(r["frog_id"], "y=", r["y"], "prob=", round(r["prob"], 4), "pred=", r["pred"])

        epoch_outputs[ep] = {
            "selected_ckpt": selected_ckpt,
            "test_file_rows": test_rows,
            "test_frog_rows": frog_rows,
            "fold_metrics": fold_metrics,
            "train_loss": train_loss_by_epoch.get(int(ep), np.nan),
        }

    return {
        "outer_test_frog": outer_test_frog,
        "train_frogs": train_frogs,
        "phase1_ckpt": phase1_ckpt,
        "epoch_outputs": epoch_outputs,
        "ckpt_dir": str(ckpt_dir),
    }


# ============================================================
# 10. full ordinary frog-LOGO run for one phase1 ckpt
# ============================================================
def run_outer_frog_logo_phase2_multi_epoch(
    meta_csv,
    npz_dir,
    phase1_ckpt,
    log_root,
    eval_epochs,
    pos_label="STP1710.7",
    neg_label="control",
    seed=0,
    window_size=500,
    window_stride=250,
    n_windows_target=32,
    phase1_patch_size=50,
    phase1_patch_stride=50,
    embed_dim=64,
    num_heads=4,
    batch_size=4,
    lr=1e-4,
    weight_decay=1e-4,
    max_epochs=70,
    freeze_window_encoder=True,
    pos_class_index=1,
):
    seed_everything_all(seed)

    df_bin, uhm2label, uhm2treat, uhm2frog = build_binary_uhm_maps(
        meta_csv,
        pos_label=pos_label,
        neg_label=neg_label,
    )

    all_npz = sorted(Path(npz_dir).glob("*.npz"))
    use_files = [
        p for p in all_npz
        if normalize_uhm_from_dataset_filename(p.name) in uhm2label
    ]

    print("n files =", len(use_files))
    for p in use_files[:10]:
        uhm = normalize_uhm_from_dataset_filename(p.name)
        print(p.name, "->", uhm, uhm2treat[uhm], uhm2label[uhm])

    frogs = sorted({frog_from_uhm_sample(normalize_uhm_from_dataset_filename(p.name)) for p in use_files})
    print("frogs:", frogs)

    all_file_rows_by_epoch = {ep: [] for ep in eval_epochs}
    all_frog_rows_by_epoch = {ep: [] for ep in eval_epochs}
    fold_summary_rows = []

    for fold_idx, outer_test_frog in enumerate(frogs):
        fold_seed = seed + fold_idx

        out = run_one_outer_fold_multi_epoch(
            outer_test_frog=outer_test_frog,
            use_files=use_files,
            uhm2label=uhm2label,
            uhm2treat=uhm2treat,
            uhm2frog=uhm2frog,
            phase1_ckpt=phase1_ckpt,
            log_root=log_root,
            eval_epochs=eval_epochs,
            seed=fold_seed,
            window_size=window_size,
            window_stride=window_stride,
            n_windows_target=n_windows_target,
            phase1_patch_size=phase1_patch_size,
            phase1_patch_stride=phase1_patch_stride,
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_size=batch_size,
            lr=lr,
            weight_decay=weight_decay,
            max_epochs=max_epochs,
            freeze_window_encoder=freeze_window_encoder,
            pos_class_index=pos_class_index,
        )

        for ep in eval_epochs:
            ep_out = out["epoch_outputs"][ep]
            train_loss = ep_out.get("train_loss", np.nan)

            # Attach the fold training loss to file/frog-level prediction rows.
            # This mirrors the random baseline fold output, where each held-out frog row has a loss value.
            file_rows_with_loss = []
            for r in ep_out["test_file_rows"]:
                rr = dict(r)
                rr["loss"] = train_loss
                file_rows_with_loss.append(rr)

            frog_rows_with_loss = []
            for r in ep_out["test_frog_rows"]:
                rr = dict(r)
                rr["loss"] = train_loss
                frog_rows_with_loss.append(rr)

            all_file_rows_by_epoch[ep].extend(file_rows_with_loss)
            all_frog_rows_by_epoch[ep].extend(frog_rows_with_loss)

            fold_summary_rows.append({
                "outer_test_frog": outer_test_frog,
                "phase1_ckpt": phase1_ckpt,
                "phase1_ckpt_id": make_phase1_ckpt_id(phase1_ckpt),
                "epoch": ep,
                "selected_ckpt": ep_out["selected_ckpt"],
                "loss": train_loss,
                "fold_ACC": ep_out["fold_metrics"]["ACC"],
                "fold_AUROC": ep_out["fold_metrics"]["AUROC"],
                "fold_AUPRC": ep_out["fold_metrics"]["AUPRC"],
                "n_test_files": len(ep_out["test_file_rows"]),
                "n_test_frogs": len(ep_out["test_frog_rows"]),
            })

    # overall metrics per epoch for this phase1 ckpt
    epoch_metric_rows = []
    epoch_dfs = {}

    for ep in eval_epochs:
        frog_rows = all_frog_rows_by_epoch[ep]
        metrics = compute_binary_metrics_from_frog_rows(frog_rows)

        df_file = pd.DataFrame([{
            "phase1_ckpt": phase1_ckpt,
            "phase1_ckpt_id": make_phase1_ckpt_id(phase1_ckpt),
            "epoch": ep,
            "file_name": r["file_name"],
            "uhm_sample": r["uhm_sample"],
            "frog_id": r["frog_id"],
            "y": r["y"],
            "prob": r["prob"],
            "pred": r["pred"],
            "loss": r.get("loss", np.nan),
        } for r in all_file_rows_by_epoch[ep]])

        df_frog = pd.DataFrame([{
            "phase1_ckpt": phase1_ckpt,
            "phase1_ckpt_id": make_phase1_ckpt_id(phase1_ckpt),
            "epoch": ep,
            "frog_id": r["frog_id"],
            "y": r["y"],
            "prob": r["prob"],
            "pred": r["pred"],
            "n_files": r["n_files"],
            "loss": r.get("loss", np.nan),
        } for r in frog_rows])

        fold_losses = [r.get("loss", np.nan) for r in fold_summary_rows if r.get("epoch") == ep]
        mean_train_loss = float(np.nanmean(fold_losses)) if len(fold_losses) > 0 and not np.all(pd.isna(fold_losses)) else np.nan

        epoch_metric_rows.append({
            "phase1_ckpt": phase1_ckpt,
            "phase1_ckpt_id": make_phase1_ckpt_id(phase1_ckpt),
            "epoch": ep,
            "ACC": metrics["ACC"],
            "AUROC": metrics["AUROC"],
            "AUPRC": metrics["AUPRC"],
            "loss": mean_train_loss,
            "n_frogs": metrics["n_frogs"],
        })

        epoch_dfs[ep] = {
            "df_file": df_file,
            "df_frog": df_frog,
            "metrics": metrics,
        }

    df_epoch_metrics = pd.DataFrame(epoch_metric_rows).sort_values("epoch").reset_index(drop=True)
    df_fold = pd.DataFrame(fold_summary_rows).sort_values(["epoch", "outer_test_frog"]).reset_index(drop=True)

    return {
        "phase1_ckpt": phase1_ckpt,
        "phase1_ckpt_id": make_phase1_ckpt_id(phase1_ckpt),
        "df_epoch_metrics": df_epoch_metrics,
        "df_fold": df_fold,
        "epoch_dfs": epoch_dfs,
    }


# ============================================================
# 11. run many phase1 ckpts and summarize over them
# ============================================================
def run_multi_phase1_multi_epoch_experiment(
    meta_csv,
    npz_dir,
    phase1_ckpt_list,
    log_root,
    eval_epochs,
    pos_label,
    neg_label,
    seed=0,
    window_size=500,
    window_stride=250,
    n_windows_target=32,
    phase1_patch_size=50,
    phase1_patch_stride=50,
    embed_dim=64,
    num_heads=4,
    batch_size=4,
    lr=1e-4,
    weight_decay=1e-4,
    max_epochs=70,
    freeze_window_encoder=True,
    pos_class_index=1,
):
    all_phase1_epoch_metrics = []
    all_phase1_fold_rows = []
    all_phase1_frog_rows = []
    all_phase1_file_rows = []

    per_phase1_results = {}

    for idx, phase1_ckpt in enumerate(phase1_ckpt_list):
        print("\n" + "=" * 120)
        print(f"[PHASE1 CKPT {idx+1}/{len(phase1_ckpt_list)}]")
        print(phase1_ckpt)
        print("=" * 120)

        phase1_id = make_phase1_ckpt_id(phase1_ckpt)
        log_root_ckpt = os.path.join(log_root, f"phase1_{phase1_id}")

        results = run_outer_frog_logo_phase2_multi_epoch(
            meta_csv=meta_csv,
            npz_dir=npz_dir,
            phase1_ckpt=phase1_ckpt,
            log_root=log_root_ckpt,
            eval_epochs=eval_epochs,
            pos_label=pos_label,
            neg_label=neg_label,
            seed=seed,
            window_size=window_size,
            window_stride=window_stride,
            n_windows_target=n_windows_target,
            phase1_patch_size=phase1_patch_size,
            phase1_patch_stride=phase1_patch_stride,
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_size=batch_size,
            lr=lr,
            weight_decay=weight_decay,
            max_epochs=max_epochs,
            freeze_window_encoder=freeze_window_encoder,
            pos_class_index=pos_class_index,
        )

        per_phase1_results[phase1_id] = results

        all_phase1_epoch_metrics.append(results["df_epoch_metrics"])
        all_phase1_fold_rows.append(results["df_fold"])

        for ep in eval_epochs:
            all_phase1_frog_rows.append(results["epoch_dfs"][ep]["df_frog"])
            all_phase1_file_rows.append(results["epoch_dfs"][ep]["df_file"])

        print("\n[phase1 summary]")
        display(results["df_epoch_metrics"])

    df_all_phase1_epoch_metrics = pd.concat(all_phase1_epoch_metrics, axis=0, ignore_index=True)
    df_all_phase1_fold = pd.concat(all_phase1_fold_rows, axis=0, ignore_index=True)
    df_all_phase1_frog = pd.concat(all_phase1_frog_rows, axis=0, ignore_index=True)
    df_all_phase1_file = pd.concat(all_phase1_file_rows, axis=0, ignore_index=True)

    df_epoch_summary = summarize_mean_std_ci95(
        df_all_phase1_epoch_metrics,
        group_col="epoch",
        metric_cols=("ACC", "AUROC", "AUPRC", "loss"),
    )

    return {
        "per_phase1_results": per_phase1_results,
        "df_all_phase1_epoch_metrics": df_all_phase1_epoch_metrics,
        "df_all_phase1_fold": df_all_phase1_fold,
        "df_all_phase1_frog": df_all_phase1_frog,
        "df_all_phase1_file": df_all_phase1_file,
        "df_epoch_summary": df_epoch_summary,
    }


# ============================================================
# 12. RUN
# ============================================================
SEED = 0   # 固定 Phase-2 seed；这里不把它当实验变量




# ---- paths ----



PROJECT_ROOT = Path("../..").resolve()
META_CSV =PROJECT_ROOT / "data" / "processed" / "metadata_with_frog.csv" 
NPZ_DIR=PROJECT_ROOT / "data" / "processed" /"mzML_npz_45"
CKPT_DIR = PROJECT_ROOT / "results" / "final" / "Task1710" / "1710_BestCKPT"
PHASE1_CKPT_LIST = [
    CKPT_DIR / "seed0_step750-loss6.684.ckpt",
    CKPT_DIR / "seed1_step350-loss6.699.ckpt",
    CKPT_DIR / "seed2_step600-loss6.676.ckpt",
    CKPT_DIR / "seed3_step700-loss6.666.ckpt",
    CKPT_DIR / "seed4_step600-loss6.675.ckpt",
    CKPT_DIR / "seed5_step400-loss6.692.ckpt",
    CKPT_DIR / "seed6_step350-loss6.700.ckpt",
    CKPT_DIR / "seed7_step300-loss6.695.ckpt",
    CKPT_DIR / "seed8_step450-loss6.679.ckpt",
    CKPT_DIR / "seed9_step300-loss6.699.ckpt",
    CKPT_DIR / "seed10_step450-loss6.721.ckpt",
    CKPT_DIR / "seed11_step550-loss6.699.ckpt",
    CKPT_DIR / "seed12_step100-loss6.711.ckpt",
    CKPT_DIR / "seed13_step700-loss6.677.ckpt",
    CKPT_DIR / "seed14_step550-loss6.698.ckpt",
    CKPT_DIR / "seed15_step500-loss6.676.ckpt",
    CKPT_DIR / "seed16_step700-loss6.667.ckpt",
    CKPT_DIR / "seed17_step550-loss6.690.ckpt",
    CKPT_DIR / "seed18_step400-loss6.697.ckpt",
    CKPT_DIR / "seed19_step450-loss6.675.ckpt",
]

LOG_ROOT = PROJECT_ROOT /"results"/"runs"/"Supervised"/"Task1710"/"pretrainedEmbeding/logs4"

POS_LABEL = "STP1710.7"
NEG_LABEL = "control"

WINDOW_SIZE = 500
WINDOW_STRIDE = 250
N_WINDOWS_TARGET = 32

PHASE1_PATCH_SIZE = 50
PHASE1_PATCH_STRIDE = 50
EMBED_DIM = 64
NUM_HEADS = 4

BATCH_SIZE = 4
LR = 1e-4
WEIGHT_DECAY = 1e-4
FREEZE_WINDOW_ENCODER = True

# 你之前已经验证过：这个 notebook 里 treatment score 取 class 0
POS_CLASS_INDEX = 0

MAX_EPOCHS = 100
EVAL_EPOCHS = [10,20,30,40,50,60,70,80,90,100]

print("DEBUG SETTINGS")
print("SEED =", SEED)
print("MAX_EPOCHS =", MAX_EPOCHS)
print("EVAL_EPOCHS =", EVAL_EPOCHS)
print("N_PHASE1_CKPTS =", len(PHASE1_CKPT_LIST))
print("POS_CLASS_INDEX =", POS_CLASS_INDEX)
print("FREEZE_WINDOW_ENCODER =", FREEZE_WINDOW_ENCODER)

results_all = run_multi_phase1_multi_epoch_experiment(
    meta_csv=META_CSV,
    npz_dir=NPZ_DIR,
    phase1_ckpt_list=PHASE1_CKPT_LIST,
    log_root=LOG_ROOT,
    eval_epochs=EVAL_EPOCHS,
    pos_label=POS_LABEL,
    neg_label=NEG_LABEL,
    seed=SEED,
    window_size=WINDOW_SIZE,
    window_stride=WINDOW_STRIDE,
    n_windows_target=N_WINDOWS_TARGET,
    phase1_patch_size=PHASE1_PATCH_SIZE,
    phase1_patch_stride=PHASE1_PATCH_STRIDE,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    batch_size=BATCH_SIZE,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    max_epochs=MAX_EPOCHS,
    freeze_window_encoder=FREEZE_WINDOW_ENCODER,
    pos_class_index=POS_CLASS_INDEX,
)

df_all_phase1_epoch_metrics = results_all["df_all_phase1_epoch_metrics"]
df_epoch_summary = results_all["df_epoch_summary"]
df_all_phase1_fold = results_all["df_all_phase1_fold"]
df_all_phase1_frog = results_all["df_all_phase1_frog"]
df_all_phase1_file = results_all["df_all_phase1_file"]

print("\n" + "=" * 120)
print("ALL PHASE1 × EPOCH METRICS")
print("=" * 120)
display(df_all_phase1_epoch_metrics.sort_values(["epoch", "phase1_ckpt_id"]).reset_index(drop=True))

print("\n" + "=" * 120)
print("FINAL EPOCH SUMMARY (mean ± 95% CI across Phase1 checkpoints)")
print("=" * 120)
display(df_epoch_summary)

# ============================================================
# 13. SAVE CSVs
# ============================================================
os.makedirs(LOG_ROOT, exist_ok=True)

out1 = os.path.join(LOG_ROOT, "all_phase1_epoch_metrics.csv")
out2 = os.path.join(LOG_ROOT, "epoch_summary_mean_std_ci95.csv")
out3 = os.path.join(LOG_ROOT, "all_phase1_fold_rows.csv")
out4 = os.path.join(LOG_ROOT, "all_phase1_frog_rows.csv")
out5 = os.path.join(LOG_ROOT, "all_phase1_file_rows.csv")

df_all_phase1_epoch_metrics.to_csv(out1, index=False)
df_epoch_summary.to_csv(out2, index=False)
df_all_phase1_fold.to_csv(out3, index=False)
df_all_phase1_frog.to_csv(out4, index=False)
df_all_phase1_file.to_csv(out5, index=False)

print("Saved:", out1)
print("Saved:", out2)
print("Saved:", out3)
print("Saved:", out4)
print("Saved:", out5)

# ============================================================
# 14. OPTIONAL: pretty formatted summary
# ============================================================
def fmt_mean_ci(mean, low, high, digits=3):
    return f"{mean:.{digits}f} ({low:.{digits}f}–{high:.{digits}f})"

df_pretty = df_epoch_summary.copy()
for m in ["ACC", "AUROC", "AUPRC", "loss"]:
    df_pretty[m] = [
        fmt_mean_ci(a, b, c)
        for a, b, c in zip(
            df_pretty[f"{m}_mean"],
            df_pretty[f"{m}_ci_low"],
            df_pretty[f"{m}_ci_high"],
        )
    ]

df_pretty = df_pretty[["epoch", "n_phase1_ckpts", "ACC", "AUROC", "AUPRC", "loss"]]
print("\nPRETTY SUMMARY")
display(df_pretty)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = df_all_phase1_epoch_metrics.copy()
metric = "AUROC"

plt.figure(figsize=(7.2, 5.2), dpi=160)

for ckpt_id, sub in df.groupby("phase1_ckpt_id"):
    sub = sub.sort_values("epoch")
    plt.plot(
        sub["epoch"],
        sub[metric],
        color="tab:blue",
        linewidth=1.4,
        alpha=0.18
    )

df_mean = (
    df.groupby("epoch", as_index=False)[metric]
      .mean()
      .sort_values("epoch")
)

plt.plot(
    df_mean["epoch"],
    df_mean[metric],
    color="tab:blue",
    linewidth=3,
    marker="o",
    markersize=6,
    label="Frozen"
)

plt.title("STP1710.7 vs control")
plt.xlabel("Epoch")
plt.ylabel(metric)
plt.xticks(sorted(df["epoch"].unique()))
plt.grid(True, alpha=0.25)
plt.legend(frameon=False)
plt.tight_layout()
plt.show()